In [1]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

'ls' is not recognized as an internal or external command,
operable program or batch file.
x train_images.pkl
x train_labels.pkl
x val_images.pkl
x val_labels.pkl


In [4]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [5]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [6]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [7]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [8]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

## **Start modified network pruning**

In [9]:
# reload full model
model = ConvNet().to(device)
model.load_state_dict(torch.load('original_model.pt'))
print("Loaded original model weights.")


Loaded original model weights.


C:\Users\nickc\AppData\Local\Temp\ipykernel_48140\3605758991.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('original_model.pt'))


In [10]:
class SlimmableConvNet(nn.Module):
    def __init__(self, linear_input_dim=1024):
        super(SlimmableConvNet, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(linear_input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)


In [11]:
def compute_flatten_dim(model, input_shape=(3, 25, 25)):
    with torch.no_grad():
        dummy_input = torch.randn(1, *input_shape)
        output = model.model[:-5](dummy_input)  # skip FC layers
        return output.view(1, -1).shape[1]

In [12]:
def get_bn_pruning_masks(model_with_bn, threshold=0.01):
    masks = []
    for module in model_with_bn.modules():
        if isinstance(module, nn.BatchNorm2d):
            gamma = module.weight.data.abs().clone()
            mask = gamma > threshold
            masks.append(mask)
    return masks

In [13]:
def prune_conv2d_weights(conv, mask_out, mask_in=None):
    W = conv.weight.data.clone()
    B = conv.bias.data.clone() if conv.bias is not None else None

    W = W[mask_out]
    if B is not None:
        B = B[mask_out]
    if mask_in is not None:
        W = W[:, mask_in, :, :]
    return W, B

In [14]:
def get_conv_layers_only(model):
    return [m for m in model.modules() if isinstance(m, nn.Conv2d)]

In [15]:
def transfer_pruned_weights(model_with_bn, model_orig, masks):
    conv_bn_layers = get_conv_layers_only(model_with_bn)
    conv_orig_layers = get_conv_layers_only(model_orig)

    mask_in = None
    for i, (conv_bn, conv_orig, mask_out) in enumerate(zip(conv_bn_layers, conv_orig_layers, masks)):
        W, B = prune_conv2d_weights(conv_bn, mask_out, mask_in)
        conv_orig.weight.data.copy_(W)
        if B is not None:
            conv_orig.bias.data.copy_(B)
        mask_in = mask_out

In [16]:
def train_one_epoch_bn(model, train_loader, optimizer, criterion, device, l1_strength=1e-4):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # add L1 penalty on batchNorm weights
        l1_loss = 0.0
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                l1_loss += torch.norm(module.weight, 1)  # gamma values

        # add the L1 loss scaled by lambda
        loss += l1_strength * l1_loss

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy


In [17]:
def train_slimmable_model(train_loader, val_loader, device, num_epochs=50):
    # Initialize model
    model = SlimmableConvNet().to(device)

    # Define optimizer and loss
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)
    criterion = nn.CrossEntropyLoss()

    # Loop through epochs
    for epoch in range(num_epochs):
        print(f"\nEpoch (bn) {epoch + 1}/{num_epochs}")

        train_loss, train_acc = train_one_epoch_bn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.2f}%")

    # Save trained model
    torch.save(model.state_dict(), "slim_model_with_bn.pth")
    print("Saved trained SlimmableConvNet for pruning.")

    return model

In [18]:
model_bn = train_slimmable_model(train_loader, val_loader, device, num_epochs=50)



Epoch (bn) 1/50


Train Loss: 1.4039, Train Acc: 39.88%
Val   Loss: 1.2221, Val   Acc: 48.95%

Epoch (bn) 2/50


Train Loss: 1.2298, Train Acc: 50.14%
Val   Loss: 1.1330, Val   Acc: 51.76%

Epoch (bn) 3/50


Train Loss: 1.1528, Train Acc: 53.94%
Val   Loss: 1.0737, Val   Acc: 56.24%

Epoch (bn) 4/50


Train Loss: 1.0981, Train Acc: 56.54%
Val   Loss: 1.0395, Val   Acc: 57.43%

Epoch (bn) 5/50


...

Train Loss: 0.4907, Train Acc: 82.38%
Val   Loss: 0.7090, Val   Acc: 75.09%

Epoch (bn) 47/50


Train Loss: 0.4834, Train Acc: 82.46%
Val   Loss: 0.6763, Val   Acc: 75.68%

Epoch (bn) 48/50


Train Loss: 0.4790, Train Acc: 82.75%
Val   Loss: 0.7014, Val   Acc: 75.29%

Epoch (bn) 49/50


Train Loss: 0.4746, Train Acc: 82.97%
Val   Loss: 0.8065, Val   Acc: 73.03%

Epoch (bn) 50/50


Train Loss: 0.4698, Train Acc: 83.01%
Val   Loss: 0.6919, Val   Acc: 76.28%
Saved trained SlimmableConvNet for pruning.


In [19]:
def count_zero_weights(model):
    total_weights = 0
    zero_weights = 0

    for param in model.parameters():
        if param.requires_grad:
            total_weights += param.numel()
            zero_weights += torch.sum(param == 0).item()

    percent_pruned = 100.0 * zero_weights / total_weights
    return total_weights, zero_weights, percent_pruned


In [20]:
def zero_out_pruned_weights(model, masks):
    conv_layers = [m for m in model.model if isinstance(m, nn.Conv2d)]
    mask_in = None

    for conv, mask_out in zip(conv_layers, masks):
        with torch.no_grad():
            # zero out pruned output channels
            out_indices = (~mask_out).nonzero(as_tuple=True)[0]
            conv.weight[out_indices] = 0
            if conv.bias is not None:
                conv.bias[out_indices] = 0

            # zero out pruned input channels if not the first layer
            if mask_in is not None:
                in_indices = (~mask_in).nonzero(as_tuple=True)[0]
                conv.weight[:, in_indices] = 0

        # save current mask as input mask for next layer
        mask_in = mask_out


In [21]:
def get_identity_masks(model_with_bn):
    masks = []
    for module in model_with_bn.model:
        if isinstance(module, nn.BatchNorm2d):
            gamma = module.weight.data
            mask = torch.ones_like(gamma).bool()  # keep everything
            masks.append(mask)
    return masks

In [22]:
model_with_bn = SlimmableConvNet()
model_with_bn.load_state_dict(torch.load("slim_model_with_bn.pth"))
model_with_bn.eval()

C:\Users\nickc\AppData\Local\Temp\ipykernel_48140\4247666759.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_with_bn.load_state_dict(torch.load("slim_model_with_bn

SlimmableConvNet(
  (model): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.25, inplace=False)
    (16): Flatten(

In [23]:

# recompute linear input dim
linear_input_dim = compute_flatten_dim(model_with_bn, input_shape=(3, 25, 25))

# original model
model_orig = ConvNet()

# Get all-True masks (simulate no pruning)
masks = get_identity_masks(model_with_bn)

# Transfer weights (no channels dropped)
transfer_pruned_weights(model_with_bn, model_orig, masks)

# Move to device
model_orig.to(device)

# Validate
val_loss, val_accuracy = validate(model_orig, val_loader, criterion, device)
print(f"No-pruning weight transfer accuracy: {val_accuracy:.2f}%, Loss: {val_loss:.4f}")


No-pruning weight transfer accuracy: 20.87%, Loss: 1.6094


In [ ]:
model_with_bn = SlimmableConvNet()
model_with_bn.load_state_dict(torch.load("slim_model_with_bn.pth"))
model_with_bn.eval()

# get slimming masks based on gamma threshold
masks = get_bn_pruning_masks(model_with_bn, threshold=0.8)

def print_pruned_channels(masks):
    for i, mask in enumerate(masks):
        total = mask.numel()
        kept = mask.sum().item()
        print(f"Layer {i+1}: Kept {int(kept)}/{total} channels ({100 * kept / total:.2f}%)")

print_pruned_channels(masks)


# recompute linear input dim
linear_input_dim = compute_flatten_dim(model_with_bn, input_shape=(3, 25, 25))
#print(linear_input_dim)

# original model
model_orig = ConvNet()

# transfer pruned Conv2d weights into original model
transfer_pruned_weights(model_with_bn, model_orig, masks)

zero_out_pruned_weights(model_orig, masks)

model_orig.to(device)

val_loss, val_accuracy = validate(model_orig, val_loader, criterion, device)
print(f"\nPruned Model Validation Accuracy: {val_accuracy:.2f}%, Loss: {val_loss:.4f}")

total, zeros, pruned_percent = count_zero_weights(model_orig)
print(f"Total weights     : {total:,}")
print(f"Zeroed weights    : {int(zeros):,}")
print(f"Percentage pruned : {pruned_percent:.2f}%")

# # 6. Save the final pruned model (optional)
# torch.save(model_orig.state_dict(), "original_model_pruned_weights.pt")
# print("Pruned weights transferred and saved.")
